# 10 — RAG: Split Treino/Validação

Divide `toldBr_full.csv` em split estratificado 80/20:
- `data/full/toldBr_train.csv` — corpus de retrieval (~16.650 tweets)
- `data/full/toldBr_val.csv` — conjunto de avaliação (~4.163 tweets)

**Atenção:** Classes raras terão poucos exemplos no val — racism (~4), xenophobia (~6), misogyny (~9). Isso é inerente ao dataset.

In [1]:
from pathlib import Path
import polars as pl
from sklearn.model_selection import train_test_split

ROOT = Path("..")
RAW = ROOT / "data" / "raw" / "toldBr_full.csv"
TRAIN_OUT = ROOT / "data" / "full" / "toldBr_train.csv"
VAL_OUT   = ROOT / "data" / "full" / "toldBr_val.csv"

df = pl.read_csv(RAW)
print(f"Total: {len(df)} tweets")
print(df["label"].value_counts().sort("label"))

Total: 21000 tweets
shape: (7, 2)
┌────────────┬───────┐
│ label      ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ homophobia ┆ 169   │
│ insult     ┆ 1502  │
│ misogyny   ┆ 44    │
│ not_toxic  ┆ 16937 │
│ obscene    ┆ 2296  │
│ racism     ┆ 21    │
│ xenophobia ┆ 31    │
└────────────┴───────┘


In [2]:
train_idx, val_idx = train_test_split(
    range(len(df)),
    test_size=0.2,
    random_state=42,
    stratify=df["label"].to_list()
)

train_df = df[list(train_idx)]
val_df   = df[list(val_idx)]

print(f"Train: {len(train_df)} | Val: {len(val_df)}")

Train: 16800 | Val: 4200


In [3]:
print("=== Distribuição Train ===")
print(train_df["label"].value_counts().sort("label"))
print("\n=== Distribuição Val ===")
print(val_df["label"].value_counts().sort("label"))

=== Distribuição Train ===
shape: (7, 2)
┌────────────┬───────┐
│ label      ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ homophobia ┆ 135   │
│ insult     ┆ 1201  │
│ misogyny   ┆ 35    │
│ not_toxic  ┆ 13550 │
│ obscene    ┆ 1837  │
│ racism     ┆ 17    │
│ xenophobia ┆ 25    │
└────────────┴───────┘

=== Distribuição Val ===
shape: (7, 2)
┌────────────┬───────┐
│ label      ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ homophobia ┆ 34    │
│ insult     ┆ 301   │
│ misogyny   ┆ 9     │
│ not_toxic  ┆ 3387  │
│ obscene    ┆ 459   │
│ racism     ┆ 4     │
│ xenophobia ┆ 6     │
└────────────┴───────┘


In [4]:
train_df.write_csv(TRAIN_OUT)
val_df.write_csv(VAL_OUT)
print(f"Salvo: {TRAIN_OUT}")
print(f"Salvo: {VAL_OUT}")

Salvo: ..\data\full\toldBr_train.csv
Salvo: ..\data\full\toldBr_val.csv
